# PI3Kα (4JPS) — Step 4: Atom-mapped reference ligand preparation

**Project:** Tryptanthrin anticancer drug discovery  
**Target:** PI3Kα / PIK3CA  
**Reference ligand:** 1LT (alpelisib)  
**Input:** `PI3Kalpha_4JPS_Step3.zip`

This notebook creates a **chemically defined SDF with the experimentally observed ligand heavy-atom positions**. It verifies the atom order by comparing the official CCD ideal atom coordinates against the ideal SDF, then transfers the **crystallographic** heavy-atom coordinates by atom name.

**Stop conditions:** any failed atom-name, element, atom-order, coordinate, or stereochemistry check halts execution. This is not a successful redocking test and does not produce a validated protein PDBQT. Protonation states still require review before docking.


In [ ]:
!pip -q install rdkit biopython numpy pandas
from google.colab import files
from pathlib import Path
import zipfile, json
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from Bio.PDB import PDBParser
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

out=Path('PI3Kalpha_4JPS_Step4')
out.mkdir(exist_ok=True)
print('Upload PI3Kalpha_4JPS_Step3.zip:')
uploaded=files.upload()
zz=[k for k in uploaded if k.lower().endswith('.zip')]
if len(zz)!=1: raise RuntimeError(f'Upload exactly one Step 3 ZIP; got {zz}')
needed={'1LT_ideal.sdf','1LT.cif','4JPS_original.pdb','4JPS_protein_AB_preliminary.pdb','pi3ka_4jps_step3_preflight.json'}
with zipfile.ZipFile(zz[0]) as z:
    if z.testzip(): raise RuntimeError('Corrupted ZIP')
    for name in needed:
        hits=[x for x in z.namelist() if x.split('/')[-1]==name]
        if len(hits)!=1: raise RuntimeError(f'{name}: expected exactly once, got {len(hits)}')
        (out/name).write_bytes(z.read(hits[0]))
print('Input files extracted:',len(needed))

In [ ]:
report3=json.loads((out/'pi3ka_4jps_step3_preflight.json').read_text())
assert report3['ccd_crystal_atom_check_passed'] is True
ccd=MMCIF2Dict(str(out/'1LT.cif'))
ids=[x.strip() for x in ccd['_chem_comp_atom.atom_id']]
elements=[x.upper() for x in ccd['_chem_comp_atom.type_symbol']]
ideal_ccd=np.array(list(zip(*[
    [float(x) for x in ccd[k]] for k in [
        '_chem_comp_atom.pdbx_model_Cartn_x_ideal',
        '_chem_comp_atom.pdbx_model_Cartn_y_ideal',
        '_chem_comp_atom.pdbx_model_Cartn_z_ideal']
])), dtype=float)
assert len(ids)==len(set(ids))

supplier=Chem.SDMolSupplier(str(out/'1LT_ideal.sdf'), removeHs=False)
mols=[m for m in supplier if m is not None]
if len(mols)!=1: raise RuntimeError('Expected one readable ideal SDF')
ideal=mols[0]
assert ideal.GetNumAtoms()==len(ids), 'CCD and SDF total atom counts differ'
coords=np.asarray(ideal.GetConformer().GetPositions())
max_ideal_deviation=float(np.linalg.norm(coords-ideal_ccd,axis=1).max())
element_matches=all(a.GetSymbol().upper()==e for a,e in zip(ideal.GetAtoms(),elements))
print('Ideal SDF vs CCD atom order maximum coordinate deviation (Å):',round(max_ideal_deviation,5))
print('Element-by-index match:',element_matches)
if max_ideal_deviation>0.02 or not element_matches:
    raise RuntimeError('Ideal SDF atom index cannot safely be mapped to CCD atom IDs')
print('PASS: CCD atom ID mapping to SDF atom indices verified')

In [ ]:
structure=PDBParser(QUIET=True).get_structure('4JPS',str(out/'4JPS_original.pdb'))
model=next(structure.get_models())
found=[(c.id,r) for c in model for r in c if r.resname.strip()=='1LT']
if len(found)!=1: raise RuntimeError('Expected exactly one crystal 1LT instance')
chain,crystal=found[0]
crystal_by_name={a.name.strip(): a for a in crystal if a.element.upper() not in {'H','D'}}
expected_heavy={n for n,e in zip(ids,elements) if e not in {'H','D'}}
if set(crystal_by_name)!=expected_heavy:
    raise RuntimeError('Crystal atom names differ from CCD heavy atom names')

# Check independent CCD model coordinates against deposited 4JPS PDB.
model_xyz_keys=['_chem_comp_atom.model_Cartn_x','_chem_comp_atom.model_Cartn_y','_chem_comp_atom.model_Cartn_z']
ccd_model=np.asarray(list(zip(*[[float(x) for x in ccd[k]] for k in model_xyz_keys])),dtype=float)
deviations=[]
for i,(name,element) in enumerate(zip(ids,elements)):
    if element not in {'H','D'}:
        a=crystal_by_name[name]
        if a.element.upper()!=element: raise RuntimeError(f'Element mismatch at {name}')
        deviations.append(float(np.linalg.norm(a.coord-ccd_model[i])))
max_crystal_ccd_deviation=max(deviations)
print('Crystal PDB vs CCD deposited model max heavy-atom difference (Å):',round(max_crystal_ccd_deviation,4))
if max_crystal_ccd_deviation>0.10:
    raise RuntimeError('CCD model and crystallographic ligand coordinates disagree: STOP')

# Preserve crystal heavy atoms; discard ideal hydrogens before re-adding them.
heavy=Chem.RemoveHs(ideal)
heavy_ids=[name for name,e in zip(ids,elements) if e not in {'H','D'}]
if heavy.GetNumAtoms()!=len(heavy_ids): raise RuntimeError('Heavy atom count after RemoveHs differs')
conf=heavy.GetConformer()
for idx,name in enumerate(heavy_ids):
    conf.SetAtomPosition(idx, [float(x) for x in crystal_by_name[name].coord])
    heavy.GetAtomWithIdx(idx).SetProp('ccd_atom_id',name)
heavy.SetProp('_Name','1LT_alpelisib_4JPS_crystal_heavy')
Chem.SanitizeMol(heavy)

# Serialize and re-read to test connectivity, stereo and atom-index order.
heavy_file=out/'1LT_crystal_heavy_atom_mapped.sdf'
writer=Chem.SDWriter(str(heavy_file)); writer.write(heavy); writer.close()
check=next((m for m in Chem.SDMolSupplier(str(heavy_file),removeHs=False) if m is not None),None)
if check is None or check.GetNumHeavyAtoms()!=30: raise RuntimeError('Written SDF failed reload')
crystal_coords_reloaded=np.asarray(check.GetConformer().GetPositions())
max_written_deviation=float(np.linalg.norm(crystal_coords_reloaded-np.array([crystal_by_name[n].coord for n in heavy_ids]),axis=1).max())
if max_written_deviation>0.003: raise RuntimeError('Written SDF did not preserve crystallographic coordinates')
print('PASS: 30 crystallographic heavy-atom coordinates retained; max deviation',round(max_written_deviation,5),'Å')
print('Heavy-atom SDF saved:',heavy_file)
print('Formula:',rdMolDescriptors.CalcMolFormula(heavy))


In [ ]:
# Build a hydrogenated reference ligand for later protonation/preparation review.
# Hydrogens are generated geometrically; heavy-atom coordinates remain unchanged.
hmol=Chem.AddHs(heavy,addCoords=True)
with_h=out/'1LT_crystal_with_generated_H_review.sdf'
w=Chem.SDWriter(str(with_h));w.write(hmol);w.close()

mapping=pd.DataFrame([{
 'sdf_heavy_atom_index_0based':i,
 'ccd_atom_name':n,
 'element':elements[ids.index(n)],
 'crystal_x_A':float(crystal_by_name[n].coord[0]),
 'crystal_y_A':float(crystal_by_name[n].coord[1]),
 'crystal_z_A':float(crystal_by_name[n].coord[2])
} for i,n in enumerate(heavy_ids)])
mapping.to_csv(out/'1LT_atom_mapping.csv',index=False)

# The CCD specifies one chiral atom, C7: S. Compare assigned CIP if interpretable.
Chem.AssignStereochemistry(heavy,cleanIt=True,force=True)
assigned=[(a.GetProp('ccd_atom_id') if a.HasProp('ccd_atom_id') else str(a.GetIdx()),a.GetProp('_CIPCode'))
          for a in heavy.GetAtoms() if a.HasProp('_CIPCode')]
print('Assigned chiral atoms from SDF:',assigned)
expected_c7=ccd['_chem_comp_atom.pdbx_stereo_config'][ids.index('C7')]
print('CCD C7 stereochemistry:',expected_c7)
if expected_c7.upper()!='S' or not any(n=='C7' and s=='S' for n,s in assigned):
    raise RuntimeError('C7 stereo validation mismatch; do not prepare docking ligand')

summary={
 'pdb_id':'4JPS','target':'PI3Kalpha','reference_ligand':'1LT',
 'ligand_common_name':'alpelisib',
 'source':'4JPS PDB coordinates and 1LT official CCD chemistry',
 'ccd_sdf_ideal_max_atom_order_deviation_A':round(max_ideal_deviation,6),
 'ccd_model_to_crystal_max_deviation_A':round(max_crystal_ccd_deviation,6),
 'sdf_written_crystal_max_deviation_A':round(max_written_deviation,6),
 'stereochemistry_assigned':assigned,
 'status':'reference ligand atom-mapped SDF completed; NOT redocked; protonation review and receptor PDBQT still pending',
 'note':'Hydrogen positions are generated; experimental heavy-atom coordinates preserved. No force-field minimization applied to reference pose.'
}
(out/'pi3ka_step4_ligand_mapping_report.json').write_text(json.dumps(summary,indent=2)+'\n')
print('PASS: reference ligand mapping and stereochemical check complete')

In [ ]:
import shutil
archive=shutil.make_archive('PI3Kalpha_4JPS_Step4','zip',out)
files.download(archive)
print('Step 4 package:',archive)
print('NEXT: receptor repair/protonation and Meeko preparation, then 1LT redocking and RMSD verification.')